# SHMS Bridge Anomaly Detection — Phase 3A (LSTM) & 3C (GNN)

**Tujuan notebook ini:**
- Melatih LSTM Autoencoder (Phase 3A) dan GNN Autoencoder (Phase 3C)
- Menggunakan **GPU T4 Colab** (jauh lebih cepat dari CPU ZBook)
- Data di-upload ke **Temporary Storage** `/content/` (bukan Google Drive)

**Yang perlu disiapkan sebelum mulai:**
- File `.npy` dari `F:\Data SHMS Batam\processed\` (upload via sel berikutnya)
- File pendukung: `processing_summary.csv`, `p2_normalizer_stats.csv`
- File kode Python: `shms_phase3a_lstm.py`, `shms_phase3c_gnn.py`, dll.

**Output yang di-download:**
- `lstm_autoencoder_best.pt` — model LSTM
- `gnn_model_best.pt` — model GNN  
- `lstm_threshold.json`, `gnn_threshold.json` — threshold P95
- Semua plot hasil

## ① Setup & Verifikasi GPU

In [ ]:
import torch, os, sys
import numpy as np
from pathlib import Path

# Cek GPU
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'GPU RAM         : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
else:
    print('⚠️  GPU tidak terdeteksi! Pastikan Runtime → Change runtime type → T4 GPU')

# Cek storage tersedia
import shutil
total, used, free = shutil.disk_usage('/content')
print(f'\nStorage /content: {free/1024**3:.0f} GB tersedia dari {total/1024**3:.0f} GB')

## ② Buat Struktur Folder di /content

In [ ]:
# Buat semua folder yang dibutuhkan
DIRS = [
    '/content/processed',      # tempat .npy files
    '/content/models',         # output model
    '/content/results/figures',# output plot
    '/content/code',           # kode Python
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f'✅ {d}')

print('\nFolder siap!')

## ③ Upload File Kode Python

Upload file-file berikut dari folder `03_code` di laptop:
- `shms_phase3a_lstm.py`
- `shms_phase3c_gnn.py`  
- `shms_config.py`
- `shms_phase2_preprocessing.py`

In [ ]:
from google.colab import files

print('Upload file kode Python (pilih semua sekaligus):')
print('  - shms_phase3a_lstm.py')
print('  - shms_phase3c_gnn.py')
print('  - shms_config.py')
print('  - shms_phase2_preprocessing.py')
uploaded = files.upload()

# Pindahkan ke /content/code/
import shutil
for fname in uploaded:
    shutil.move(fname, f'/content/code/{fname}')
    print(f'✅ {fname} → /content/code/')

# Tambahkan ke sys.path
sys.path.insert(0, '/content/code')
print('\nsys.path updated')

## ④ Patch shms_config.py untuk Colab

Path di config masih mengarah ke `F:\Data SHMS Batam\...` (Windows).
Kita override dengan path `/content/` (Linux Colab).

In [ ]:
# Baca config asli
config_path = '/content/code/shms_config.py'
with open(config_path, 'r') as f:
    config_src = f.read()

# Override path ke /content/ — tambahkan di akhir file
colab_overrides = '''

# ══════════════════════════════════════════════
# COLAB OVERRIDE — ditambahkan otomatis
# Mengarahkan semua path ke /content/ (Temporary Storage)
# ══════════════════════════════════════════════
from pathlib import Path as _Path

# Override semua path
OUT_DATA_OVERRIDE  = _Path('/content/processed')
RAW_DATA_OVERRIDE  = _Path('/content/processed')  # tidak dipakai, hanya untuk compat
ABN_DATA_OVERRIDE  = _Path('/content/processed')  # tidak dipakai
MODEL_DIR          = _Path('/content/models')
RESULTS_DIR        = _Path('/content/results')
DATA_PROCESSED_DIR = _Path('/content/processed')
PROJECT_ROOT       = _Path('/content')

def get_processed_dir():
    p = _Path('/content/processed')
    p.mkdir(parents=True, exist_ok=True)
    return p

def get_raw_dir():
    return _Path('/content/processed')

def get_abnormal_dir():
    return _Path('/content/processed')
'''

# Tulis ulang config
with open(config_path, 'w') as f:
    f.write(config_src + colab_overrides)

print('✅ shms_config.py di-patch untuk Colab')
print('   Semua path diarahkan ke /content/')

# Test import
import importlib
import shms_config
importlib.reload(shms_config)
print(f'\nProcessed dir : {shms_config.get_processed_dir()}')
print(f'Model dir     : {shms_config.MODEL_DIR}')
print(f'Results dir   : {shms_config.RESULTS_DIR}')
print(f'Main channels : {len(shms_config.MAIN_CHANNELS)} channel')

## ⑤ Upload File Pendukung (CSV)

Upload dari `F:\Data SHMS Batam\processed\`:
- `processing_summary.csv`
- `p2_normalizer_stats.csv`
- `p2_correlation_matrix.csv` (untuk GNN)

In [ ]:
print('Upload file CSV pendukung:')
print('  - processing_summary.csv')
print('  - p2_normalizer_stats.csv')
print('  - p2_correlation_matrix.csv')
uploaded_csv = files.upload()

for fname in uploaded_csv:
    shutil.move(fname, f'/content/processed/{fname}')
    print(f'✅ {fname} → /content/processed/')

# Verifikasi
import pandas as pd
summary = pd.read_csv('/content/processed/processing_summary.csv')
print(f'\nprocessing_summary.csv:')
print(f'  Total hari    : {len(summary)}')
print(f'  Split train   : {(summary["split"]=="train").sum()} hari')
print(f'  Split val     : {(summary["split"]=="val").sum()} hari')
print(f'  Split test    : {(summary["split"]=="test").sum()} hari')
print(f'  Status OK     : {(summary["status"]=="ok").sum()} hari')

## ⑥ Upload File .npy

**Ini bagian terbesar (~46 GB total).**

Cara yang disarankan — pilih salah satu:

**Opsi A (Mudah): Upload langsung via browser**
- Pilih semua `*_X.npy` dan `*_y.npy` dari `F:\Data SHMS Batam\processed\`
- Upload sekaligus (browser handle multiple file upload)
- Estimasi waktu: tergantung kecepatan internet

**Opsi B (Lebih cepat): Zip per-batch lalu upload**
- Di Windows: compress beberapa hari sekaligus → upload zip → extract
- Sel di bawah ada fungsi extract zip otomatis

> **Catatan**: Hanya perlu upload hari TRAIN (25 hari) dan VAL (10 hari).
> Hari TEST tidak dipakai untuk training — bisa skip.

In [ ]:
# CEK dulu hari mana yang perlu di-upload
summary = pd.read_csv('/content/processed/processing_summary.csv')
needed = summary[summary['split'].isin(['train','val']) & (summary['status']=='ok')]['date'].tolist()

print(f'Hari yang perlu di-upload: {len(needed)} hari')
print(f'Train: {sum(1 for d in summary[summary["split"]=="train"]["date"] if int(d) in [int(x) for x in needed])} hari')
print(f'Val  : {sum(1 for d in summary[summary["split"]=="val"]["date"] if int(d) in [int(x) for x in needed])} hari')
print()
print('File yang dibutuhkan:')
for d in sorted(needed)[:5]:
    print(f'  {d}_X.npy  (~1.3 GB)')
    print(f'  {d}_y.npy  (~17 KB)')
print(f'  ... dan {len(needed)-5} hari lainnya')

In [ ]:
# UPLOAD .npy files
# Jalankan sel ini berulang kali jika perlu upload batch-per-batch

print('Upload file .npy (boleh bertahap, jalankan sel ini berkali-kali):')
uploaded_npy = files.upload()

for fname in uploaded_npy:
    dest = f'/content/processed/{fname}'
    shutil.move(fname, dest)
    size_mb = Path(dest).stat().st_size / 1024**2
    print(f'✅ {fname} ({size_mb:.0f} MB)')

In [ ]:
# OPSI B: Jika upload via ZIP — extract di sini
import zipfile

def extract_npy_zip(zip_filename):
    """Extract zip berisi .npy files ke /content/processed/"""
    print(f'Extracting {zip_filename}...')
    with zipfile.ZipFile(zip_filename, 'r') as zf:
        for name in zf.namelist():
            if name.endswith('.npy'):
                zf.extract(name, '/content/processed/')
                print(f'  ✅ {name}')
    os.remove(zip_filename)
    print('Selesai!')

# Contoh pemakaian:
# extract_npy_zip('batch_train_1.zip')

In [ ]:
# VERIFIKASI: cek file .npy yang sudah ada
npy_files  = list(Path('/content/processed').glob('*_X.npy'))
total_size = sum(f.stat().st_size for f in npy_files) / 1024**3

print(f'File .npy tersedia : {len(npy_files)} hari')
print(f'Total size         : {total_size:.1f} GB')
print()

# Cek mana yang masih kurang
summary  = pd.read_csv('/content/processed/processing_summary.csv')
needed   = set(str(d) for d in summary[
    summary['split'].isin(['train','val']) & (summary['status']=='ok')
]['date'].tolist())
uploaded = set(f.stem.replace('_X','') for f in npy_files)
missing  = needed - uploaded

if missing:
    print(f'⚠️  {len(missing)} hari belum ter-upload:')
    for d in sorted(missing):
        print(f'   {d}_X.npy')
else:
    print('✅ Semua file sudah ter-upload! Siap training.')

# Cek storage sisa
total, used, free = shutil.disk_usage('/content')
print(f'\nStorage tersisa: {free/1024**3:.1f} GB')

## ⑦ Phase 3A — Training LSTM Autoencoder

In [ ]:
# Pastikan module sudah di-load ulang setelah patch
for mod in list(sys.modules.keys()):
    if 'shms' in mod:
        del sys.modules[mod]
sys.path.insert(0, '/content/code')

import shms_config
import shms_phase3a_lstm as p3a
import importlib
importlib.reload(shms_config)
importlib.reload(p3a)

print('Module loaded:')
print(f'  Processed dir : {shms_config.get_processed_dir()}')
print(f'  Model dir     : {shms_config.MODEL_DIR}')
print(f'  GPU available : {torch.cuda.is_available()}')

In [ ]:
# Override HP untuk GPU Colab — batch size lebih besar, lebih efisien
HP_COLAB = {
    **p3a.HP,             # ambil semua default dari ZBook
    'batch_size'       : 256,    # GPU bisa handle batch lebih besar
    'max_train_windows': 100_000, # 2x lebih banyak dari ZBook (GPU lebih cepat)
    'n_epochs'         : 100,
    'patience'         : 15,     # lebih sabar karena GPU lebih konsisten
}

print('Hyperparameter Colab:')
for k, v in HP_COLAB.items():
    default = p3a.HP.get(k, 'NEW')
    mark = ' ← diubah' if v != default else ''
    print(f'  {k:<22}: {v}{mark}')

In [ ]:
import time
t_start = time.time()

print('='*65)
print('  MULAI TRAINING LSTM AUTOENCODER')
print('='*65)

model_lstm, history = p3a.train(
    hp        = HP_COLAB,
    save_dir  = shms_config.MODEL_DIR,
    results_dir = shms_config.RESULTS_DIR,
)

t_elapsed = time.time() - t_start
print(f'\n✅ Training selesai dalam {t_elapsed/60:.1f} menit')

In [ ]:
# Kalibrasi threshold dari data validation
print('Kalibrasi threshold...')
threshold_lstm = p3a.calibrate_threshold(
    model       = model_lstm,
    hp          = HP_COLAB,
    save_dir    = shms_config.MODEL_DIR,
    results_dir = shms_config.RESULTS_DIR,
)
print(f'\n✅ Threshold LSTM: {threshold_lstm:.6f}')

## ⑧ Phase 3C — Training GNN Autoencoder

In [ ]:
# Install PyTorch Geometric (dibutuhkan GNN)
import subprocess
torch_ver = torch.__version__.split('+')[0]
cuda_ver  = 'cu121' if torch.cuda.is_available() else 'cpu'

print(f'Installing PyG untuk torch {torch_ver} + {cuda_ver}...')
subprocess.run([
    'pip', 'install', '-q',
    'torch-geometric',
    f'--extra-index-url',
    f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html'
], check=True)
print('✅ PyG installed')

In [ ]:
# Reload modules setelah install PyG
for mod in list(sys.modules.keys()):
    if 'shms' in mod or 'torch_geometric' in mod:
        del sys.modules[mod]

import shms_config, importlib
importlib.reload(shms_config)
import shms_phase3c_gnn as p3c
importlib.reload(p3c)

print(f'PyG available : {p3c.PYG_AVAILABLE}')
print(f'Torch avail   : {p3c.TORCH_AVAILABLE}')
print(f'Graph nodes   : {p3c.N_NODES}')
print(f'Graph edges   : akan dihitung dari p2_correlation_matrix.csv')

In [ ]:
HP_GNN_COLAB = {
    **p3c.HP,
    'batch_size'       : 256,
    'max_train_windows': 100_000,
    'n_epochs'         : 100,
    'patience'         : 15,
}

print('Hyperparameter GNN Colab:')
for k, v in HP_GNN_COLAB.items():
    default = p3c.HP.get(k, 'NEW')
    mark = ' ← diubah' if v != default else ''
    print(f'  {k:<22}: {v}{mark}')

In [ ]:
t_start = time.time()

print('='*65)
print('  MULAI TRAINING GNN AUTOENCODER')
print('='*65)

model_gnn, graph_data, history_gnn = p3c.train(
    hp          = HP_GNN_COLAB,
    save_dir    = shms_config.MODEL_DIR,
    results_dir = shms_config.RESULTS_DIR,
)

t_elapsed = time.time() - t_start
print(f'\n✅ GNN Training selesai dalam {t_elapsed/60:.1f} menit')

In [ ]:
# Kalibrasi threshold GNN
print('Kalibrasi threshold GNN...')
threshold_gnn = p3c.calibrate_threshold(
    model       = model_gnn,
    graph_data  = graph_data,
    hp          = HP_GNN_COLAB,
    save_dir    = shms_config.MODEL_DIR,
    results_dir = shms_config.RESULTS_DIR,
)
print(f'\n✅ Threshold GNN: {threshold_gnn:.6f}')

## ⑨ Download Hasil ke Laptop

In [ ]:
import zipfile, os
from google.colab import files

# Kumpulkan semua output dalam 1 zip
output_zip = '/content/shms_models_output.zip'

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:

    # Model files
    model_files = [
        '/content/models/lstm_autoencoder_best.pt',
        '/content/models/lstm_threshold.json',
        '/content/models/lstm_hp.json',
        '/content/models/gnn_model_best.pt',
        '/content/models/gnn_threshold.json',
        '/content/models/gnn_hp.json',
        '/content/models/gnn_graph_data.pkl',
    ]
    for fpath in model_files:
        if Path(fpath).exists():
            zf.write(fpath, f'models/{Path(fpath).name}')
            print(f'✅ {Path(fpath).name}')

    # Plot figures
    for fig in Path('/content/results/figures').glob('*.png'):
        zf.write(str(fig), f'figures/{fig.name}')
        print(f'✅ figures/{fig.name}')

    # Metrics CSV
    for csv in Path('/content/results').glob('*.csv'):
        zf.write(str(csv), f'results/{csv.name}')

zip_size = Path(output_zip).stat().st_size / 1024**2
print(f'\nZip size: {zip_size:.0f} MB')
print('Mendownload...')
files.download(output_zip)

## ⑩ Setelah Download — Letakkan di ZBook

Extract `shms_models_output.zip` lalu:

```
models/
  lstm_autoencoder_best.pt  → D:\Mine\Penelitian\2025\shms_03_code_zbook\04_models\
  lstm_threshold.json       → D:\Mine\Penelitian\2025\shms_03_code_zbook\04_models\
  lstm_hp.json              → D:\Mine\Penelitian\2025\shms_03_code_zbook\04_models\
  gnn_model_best.pt         → D:\Mine\Penelitian\2025\shms_03_code_zbook\04_models\
  gnn_threshold.json        → D:\Mine\Penelitian\2025\shms_03_code_zbook\04_models\
  gnn_graph_data.pkl        → D:\Mine\Penelitian\2025\shms_03_code_zbook\04_models\

figures/ → D:\Mine\Penelitian\2025\shms_03_code_zbook\05_results\figures\
```

Kemudian jalankan Phase 4 & 5 di ZBook (data sudah ada, tidak perlu upload lagi):
```
python run_pipeline.py --phase 4 5
```